# A2. Leave-one-year-out (LOYO) evaluation, persistence baseline and the SI metrics table

**Reporting convention (decided 2026-09-21): the headline number is the plain R² of the pooled
out-of-fold predictions ("R²")**, the same quantity the feature-selection pipeline and the persistence
baseline use. Everything else -- the mean of per-year R², RMSE, mean bias, normalised mean bias, Pearson r,
Spearman rho, index of agreement, FAC2 and the categorical hit / false-alarm / correctness scores -- goes
to the supplement (`Paper/figures/data/SI_metrics_table.csv`, `Paper/tables_SI.md`).

Method: the functional form of each lead equation is fixed as found by PySR; its numeric constants are
re-fitted by nonlinear least squares on n-1 years and the equation is scored on the held-out year.
Persistence = linear regression on the target's own lag-1, re-fitted per held-out year, evaluated on the
same days. The neural-net pipeline is the same MLP used in feature selection.

In [1]:
# Setup
import sys
from pathlib import Path
HERE = Path.cwd().resolve()
sys.path.insert(0, str(HERE.parent)); sys.path.insert(0, str(HERE.parents[1]))
import json
import numpy as np, pandas as pd
from scipy.optimize import curve_fit
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from style import ROOT, DATA_OUT
DATA = ROOT / "Data/processed/colombia"

In [2]:
# Settings (edit here)
EVENT_PERCENTILES = (90, 95)          # events for the categorical scores (hit rate, false-alarm ratio, POC)
SAVE = True

In [3]:
# The two lead equations (constants are refitted per fold; these are the full-data starting values)
# 2026-10-20: after the fire-file correction the PM2.5 lead is complexity 13 of the constrained front, seed 0
# (07_units_fire2_equation_discovery):  PM2.5 = sqrt(a * (fire_E_lag2 + fire_N)) + b * CO / sqrt(wind speed)
def pm25_model(X, a, b):
    co, wind_speed_era5, fire_n, fire_e2 = X
    return np.sqrt(np.clip(a * (fire_e2 + fire_n), 0, None)) + b * co / np.sqrt(wind_speed_era5)

# 2026-10-20 (corrected fire file): the O3 lead is complexity 15, seed 0, of the constrained front with PM2.5, temperature and
# the raw factors of the selected variables offered (07_o3_units_fire2_pm_equation_discovery):
#   O3 = sqrt( a * SW * sqrt(PM2.5 * NO2 / NO) + b * F_E,-2 ).
def o3_model(X, a, b):
    no, no2, sw, pm25, fire_e2 = X
    return np.sqrt(np.clip(a * sw * np.sqrt(np.clip(pm25 * no2 / no, 0, None)) + b * fire_e2, 0, None))

SPECS = {
    "pm25": dict(model=pm25_model, master=DATA / "bogota/bogota_master_dataset.parquet",
                 features=["co", "wind_speed_era5", "hotspots_sector_N", "hotspots_sector_E_lag2"], p0=[0.10428376, 15.679531]),
    "o3": dict(model=o3_model, master=DATA / "bogota/bogota_o3_master_dataset.parquet",
               features=["no", "no2", "radiation_mean", "pm25", "hotspots_sector_E_lag2"], p0=[1 / 2.034343, 0.10787038],
               # scored on the days the free-form equation also has, so every O3 comparison is on identical rows (the days on which every candidate of the ozone search is present)
               sample_features=["no2", "no", "radiation_mean", "hotspots_sector_E_lag2", "rh_2m", "wind_speed_era5", "w700", "temp_2m", "pm25"]),
}
def load(p):
    d = pd.read_parquet(p); d["date"] = pd.to_datetime(d["date"]); d["year"] = d["date"].dt.year; return d

In [4]:
# 1. Lead equations, leave one year out
rows, summ, preds = [], [], {}
for pol, s in SPECS.items():
    d = load(s["master"]).dropna(subset=s["features"] + s.get("sample_features", []) + [pol]).reset_index(drop=True)
    X = tuple(d[c].to_numpy(float) for c in s["features"]); y = d[pol].to_numpy(float)
    popt_full, _ = curve_fit(s["model"], X, y, p0=s["p0"], maxfev=50000)
    pred_in = s["model"](X, *popt_full); pred = np.full(len(d), np.nan)
    for yr in sorted(d.year.unique()):
        te = (d.year == yr).to_numpy()
        try: popt, _ = curve_fit(s["model"], tuple(a[~te] for a in X), y[~te], p0=popt_full, maxfev=50000)
        except RuntimeError: popt = popt_full
        pred[te] = s["model"](tuple(a[te] for a in X), *popt)
        rows.append(dict(pollutant=pol, year=int(yr), r2=r2_score(y[te], pred[te]), n=int(te.sum()),
                         rmse=float(np.sqrt(np.mean((pred[te] - y[te]) ** 2))), mean_bias=float(np.mean(pred[te] - y[te]))))
    by = pd.DataFrame([r for r in rows if r["pollutant"] == pol])
    summ.append(dict(pollutant=pol, r2_pooled=r2_score(y, pred), mean_of_years_r2=by.r2.mean(), insample_r2=r2_score(y, pred_in), n=len(d), n_years=len(by),
                     worst_year=int(by.loc[by.r2.idxmin(), "year"]), worst_r2=by.r2.min(), best_year=int(by.loc[by.r2.idxmax(), "year"]), best_r2=by.r2.max()))
    preds[pol] = pd.DataFrame(dict(date=d.date, year=d.year, obs=y, pred_loyo=pred, pred_insample=pred_in))
by_year = pd.DataFrame(rows); summary = pd.DataFrame(summ).rename(columns={"r2_pooled": "pooled_r2"})
if SAVE:
    by_year.to_csv(DATA_OUT / "loyo_by_year.csv", index=False)
    summary.to_csv(DATA_OUT / "loyo_summary.csv", index=False)
    for pol, p in preds.items(): p.to_csv(DATA_OUT / f"loyo_predictions_{pol}.csv", index=False)
summary.round(4)

,pollutant,pooled_r2,mean_of_years_r2,insample_r2,n,n_years,worst_year,worst_r2,best_year,best_r2
0,pm25,0.6768,0.6548,0.6849,3283,9,2014,0.3827,2015,0.7535
1,o3,0.6030,0.5359,0.6190,2917,8,2017,0.2656,2016,0.6907


In [5]:
# 2. Persistence on identical rows, and the year-by-year comparison
common, by_rows, pers_preds, pers_raw = [], [], {}, {}
for pol, s in SPECS.items():
    d = load(s["master"]); sub = d.dropna(subset=s["features"] + s.get("sample_features", []) + [pol, f"{pol}_lag1"]).reset_index(drop=True)
    y = sub[pol].to_numpy(float); x = sub[[f"{pol}_lag1"]].to_numpy(float); pp = np.full(len(sub), np.nan)
    eq = preds[pol].set_index("date").loc[sub.date]
    per_p, per_e = [], []
    for yr in sorted(sub.year.unique()):
        te = (sub.year == yr).to_numpy()
        pp[te] = LinearRegression().fit(x[~te], y[~te]).predict(x[te])
        e = eq[eq.year == yr]
        per_p.append(r2_score(y[te], pp[te])); per_e.append(r2_score(e.obs, e.pred_loyo))
        by_rows.append(dict(pollutant=pol, year=int(yr), n=int(te.sum()), persistence_r2=per_p[-1], equation_r2=per_e[-1], equation_minus_persistence=per_e[-1] - per_p[-1]))
    common.append(dict(pollutant=pol, n_common=len(sub), persistence_pooled=r2_score(y, pp), persistence_mean_of_years=float(np.mean(per_p)),
                       equation_pooled=r2_score(eq.obs, eq.pred_loyo), equation_mean_of_years=float(np.mean(per_e))))
    pers_preds[pol] = pd.DataFrame(dict(date=sub.date, year=sub.year, obs=y, pred=pp))
    pers_raw[pol] = pd.DataFrame(dict(date=sub.date, year=sub.year, obs=y, pred=x[:, 0]))   # yesterday's value itself as the forecast
common = pd.DataFrame(common); by_rows = pd.DataFrame(by_rows)
if SAVE:
    common.to_csv(DATA_OUT / "equation_vs_persistence_common_rows.csv", index=False)
    for pol, p in pers_preds.items(): p.to_csv(DATA_OUT / f"persistence_predictions_{pol}.csv", index=False)   # per-row, for the paired bootstrap (Fig 8, R2/R7)
    by_rows.to_csv(DATA_OUT / "equation_vs_persistence_by_year.csv", index=False)
common.round(4)

,pollutant,n_common,persistence_pooled,persistence_mean_of_years,equation_pooled,equation_mean_of_years
0,pm25,3283,0.5686,0.5156,0.6768,0.6548
1,o3,2915,0.5850,0.5251,0.6032,0.5363


In [6]:
# 3. Neural-net pipeline (same MLP as feature selection), with the year kept so both R2 aggregations can be computed
def nn_loyo(d, feats, target):
    sub = d.dropna(subset=feats + [target]).reset_index(drop=True); X = sub[feats].to_numpy(); y = sub[target].to_numpy(); pr = np.full(len(sub), np.nan)
    for yr in sorted(sub.year.unique()):
        te = (sub.year == yr).to_numpy()
        if te.sum() < 10 or (~te).sum() < 50: continue
        sc = StandardScaler().fit(X[~te])
        m = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", solver="adam", early_stopping=True, n_iter_no_change=10, max_iter=500, random_state=0)
        m.fit(sc.transform(X[~te]), y[~te]); pr[te] = m.predict(sc.transform(X[te]))
    ok = ~np.isnan(pr)
    return pd.DataFrame(dict(date=sub.date[ok], year=sub.year[ok], obs=y[ok], pred=pr[ok]))
NN = {}
for city, pol, master, sel in [("Bogota", "pm25", "bogota/bogota_master_dataset.parquet", "bogota/bogota_feature_selection.json"),
                               ("Bogota", "o3", "bogota/bogota_o3_master_dataset.parquet", "bogota/bogota_o3_feature_selection.json"),
                               ("Medellin", "pm25", "medellin/medellin_master_dataset.parquet", "medellin/medellin_feature_selection.json"),
                               ("Medellin", "o3", "medellin/medellin_o3_master_dataset.parquet", "medellin/medellin_o3_feature_selection.json")]:
    NN[(city, pol)] = nn_loyo(load(DATA / master), json.load(open(DATA / sel))["selected_no_lag"], pol)
{k: round(r2_score(v.obs, v.pred), 3) for k, v in NN.items()}

{('Bogota', 'pm25'): 0.724,
 ('Bogota', 'o3'): 0.658,
 ('Medellin', 'pm25'): 0.508,
 ('Medellin', 'o3'): 0.428}

In [7]:
# 4. The SI metrics table (all metrics, every model)
def ioa(o, p):
    om = o.mean(); return 1 - np.sum((p - o) ** 2) / np.sum((np.abs(p - om) + np.abs(o - om)) ** 2)
def metrics(df, obs="obs", pred="pred"):
    o, p = df[obs].to_numpy(float), df[pred].to_numpy(float); mb = float(np.mean(p - o))
    m = dict(n=len(o), r2_pooled=r2_score(o, p), r2_mean_of_years=float(np.mean([r2_score(g[obs], g[pred]) for _, g in df.groupby("year")])),
             pearson_r=float(pearsonr(o, p)[0]), spearman_rho=float(spearmanr(o, p)[0]), rmse=float(np.sqrt(np.mean((p - o) ** 2))),
             mean_bias=mb, nmb_pct=100 * mb / o.mean(), ioa=float(ioa(o, p)), fac2=float(np.mean((p / np.where(o == 0, np.nan, o) >= 0.5) & (p / np.where(o == 0, np.nan, o) <= 2))))
    for q in EVENT_PERCENTILES:
        thr = np.percentile(o, q); ev, pe = o >= thr, p >= thr; tp, fn, fp = int((ev & pe).sum()), int((ev & ~pe).sum()), int((~ev & pe).sum())
        m[f"hit_rate_p{q}"] = tp / (tp + fn); m[f"false_alarm_ratio_p{q}"] = fp / (tp + fp) if tp + fp else np.nan
        m[f"csi_p{q}"] = tp / (tp + fn + fp); m[f"poc_p{q}"] = float(np.mean(ev == pe))   # proportion correct
    return m
recs = []
for pol in ("pm25", "o3"):
    eq = preds[pol].rename(columns={"pred_loyo": "pred"})
    recs.append(dict(city="Bogota", pollutant=pol, model="lead equation", **metrics(eq)))
    recs.append(dict(city="Bogota", pollutant=pol, model="persistence, linear fit (same days)", **metrics(pers_preds[pol])))
    recs.append(dict(city="Bogota", pollutant=pol, model="persistence, yesterday's value as forecast", **metrics(pers_raw[pol])))
for (city, pol), v in NN.items():
    recs.append(dict(city=city, pollutant=pol, model="neural-net pipeline", **metrics(v)))
si = pd.DataFrame(recs).sort_values(["city", "pollutant", "model"]).reset_index(drop=True)
if SAVE:
    si.to_csv(DATA_OUT / "SI_metrics_table.csv", index=False)
    (ROOT / "Paper" / "tables_SI.md").write_text(
        "# SI table: all metrics for every model (generated by notebook A2; do not hand-edit)\n\n"
        "R2 pooled = R2 of all out-of-fold predictions (the headline number in the paper). R2 mean of years = mean over held-out years. "
        "Events are days at or above the observed 90th/95th percentile; POC = proportion of days correctly classified event / non-event. "
        "Equation and persistence are scored on identical Bogotá rows; the neural-net pipeline uses its own feature-selected rows.\n\n"
        + si.round(3).to_markdown(index=False) + "\n")
si.round(3)

,city,pollutant,model,n,r2_pooled,r2_mean_of_years,pearson_r,spearman_rho,rmse,mean_bias,...,ioa,fac2,hit_rate_p90,false_alarm_ratio_p90,csi_p90,poc_p90,hit_rate_p95,false_alarm_ratio_p95,csi_p95,poc_p95
0,Bogota,o3,lead equation,2917,0.603,0.536,0.777,0.756,4.408,0.072,...,0.860,0.995,0.349,0.244,0.314,0.924,0.253,0.275,0.231,0.958
1,Bogota,o3,neural-net pipeline,2917,0.658,0.603,0.812,0.804,4.091,-0.012,...,0.892,0.998,0.432,0.296,0.365,0.925,0.301,0.397,0.251,0.955
2,Bogota,o3,"persistence, linear fit (same days)",2915,0.585,0.525,0.765,0.757,4.504,-0.000,...,0.855,0.993,0.370,0.275,0.324,0.923,0.226,0.365,0.200,0.955
3,Bogota,o3,"persistence, yesterday's value as forecast",2915,0.533,0.463,0.767,0.759,4.778,-0.001,...,0.874,0.988,0.562,0.438,0.390,0.912,0.527,0.476,0.356,0.952
4,Bogota,pm25,lead equation,3283,0.677,0.655,0.823,0.834,4.564,-0.049,...,0.899,0.988,0.398,0.284,0.344,0.924,0.267,0.429,0.222,0.953
5,Bogota,pm25,neural-net pipeline,3283,0.724,0.707,0.851,0.849,4.218,-0.060,...,0.916,0.991,0.480,0.288,0.402,0.928,0.327,0.407,0.267,0.955
6,Bogota,pm25,"persistence, linear fit (same days)",3283,0.569,0.516,0.754,0.743,5.272,-0.004,...,0.847,0.978,0.304,0.333,0.264,0.915,0.212,0.327,0.192,0.955
7,Bogota,pm25,"persistence, yesterday's value as forecast",3283,0.511,0.446,0.755,0.744,5.614,-0.001,...,0.866,0.970,0.538,0.462,0.368,0.907,0.485,0.515,0.320,0.948
8,Medellin,o3,neural-net pipeline,1389,0.428,0.418,0.659,0.616,10.787,-0.458,...,0.787,0.995,0.230,0.360,0.204,0.910,0.157,0.312,0.147,0.954
9,Medellin,pm25,neural-net pipeline,2012,0.508,0.225,0.714,0.614,4.936,0.228,...,0.823,0.994,0.450,0.283,0.382,0.927,0.416,0.288,0.356,0.962
